# BurnInWatch — Data Understanding & Raw Data Extraction

We now know the NASA package contains:
- 120 CSV files from SMU characterization of new devices
- 40 TXT metadata files
- 12 MATLAB aging/thermal-overstress files
- 1 Excel aging log
- 4 documentation files

The first MATLAB aging experiment already exposes synchronized signals including gate voltage, collector voltage, heat-sink temperature, gate current, collector current, time, and package temperature.

**This notebook is still exploratory. We will NOT train a model or invent labels yet.**

Main goal:
1. inspect the real CSV schemas
2. inspect the nested MATLAB `measurement` objects
3. inspect the aging log
4. identify device IDs, time axes, and degradation measurements
5. create a first raw-data map that will guide the canonical CSV

In [1]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

DATA_ROOT = Path("dataset/Data").resolve()

print("DATA_ROOT:", DATA_ROOT)
print("Exists:", DATA_ROOT.exists())

DATA_ROOT: C:\Users\Abhishek Kuma\Desktop\BurnInWatch\dataset\Data
Exists: True


## 1. What we already learned

The discovery notebook found **177 files** in total: 120 CSV, 40 TXT, 12 MAT, 4 DOC and 1 XLS. fileciteturn0file0L187-L195

The dataset has several distinct sources. In particular, the aging section contains one large DC-gate MATLAB experiment, one large square-signal experiment, and device-specific aging files for Devices 2–5. fileciteturn0file0L170-L184

The DC-gate MATLAB file contains 301,680 samples for each of the main signals: gate voltage, collector voltage, heat-sink temperature, gate current, collector current, time, and package temperature. fileciteturn0file0L208-L208

In [2]:
# ============================================================
# 2. INSPECT REPRESENTATIVE SMU CSV FILES
# ============================================================

csv_files = sorted(DATA_ROOT.rglob("*.csv"))
print("CSV files:", len(csv_files))

# Show a few files from different device/part folders.
for p in csv_files[:10]:
    print(" -", p.relative_to(DATA_ROOT))

print("\nInspecting first few CSVs...")
csv_schema_rows = []

for path in csv_files[:12]:
    try:
        df = pd.read_csv(path)
        csv_schema_rows.append({
            "file": str(path.relative_to(DATA_ROOT)),
            "rows": len(df),
            "columns": len(df.columns),
            "column_names": " | ".join(map(str, df.columns)),
            "dtypes": " | ".join(f"{c}:{df[c].dtype}" for c in df.columns)
        })
        print("\n", path.relative_to(DATA_ROOT))
        print("shape:", df.shape)
        print("columns:", list(df.columns))
        display(df.head(3))
    except Exception as e:
        print("FAILED:", path, "|", repr(e))

csv_schema_df = pd.DataFrame(csv_schema_rows)
display(csv_schema_df)
csv_schema_df.to_csv("smu_csv_schema_sample.csv", index=False)

CSV files: 120
 - SMU Data for new devices\IGBT-IRG4BC30K\Part 11\Breakdown.csv
 - SMU Data for new devices\IGBT-IRG4BC30K\Part 11\LeakageIV.csv
 - SMU Data for new devices\IGBT-IRG4BC30K\Part 11\Turn On.csv
 - SMU Data for new devices\IGBT-IRG4BC30K\Part 12\Breakdown.csv
 - SMU Data for new devices\IGBT-IRG4BC30K\Part 12\LeakageIV.csv
 - SMU Data for new devices\IGBT-IRG4BC30K\Part 12\Turn On.csv
 - SMU Data for new devices\IGBT-IRG4BC30K\Part 13\Breakdown.csv
 - SMU Data for new devices\IGBT-IRG4BC30K\Part 13\LeakageIV.csv
 - SMU Data for new devices\IGBT-IRG4BC30K\Part 13\Turn On.csv
 - SMU Data for new devices\IGBT-IRG4BC30K\Part 14\Breakdown.csv

Inspecting first few CSVs...

 SMU Data for new devices\IGBT-IRG4BC30K\Part 11\Breakdown.csv
shape: (199, 2)
columns: ['5.452415E+2', '1.042480E-5']


,5.452415E+2,1.042480E-5
0,550.5257,0.000004
1,551.0267,0.000004
2,551.5286,0.000004



 SMU Data for new devices\IGBT-IRG4BC30K\Part 11\LeakageIV.csv
shape: (199, 2)
columns: ['1.007554E+0', '3.848548E-9']


,1.007554E+0,3.848548E-9
0,4.011488,1.027833e-08
1,7.021188,1.868011e-08
2,10.063710,2.692541e-08



 SMU Data for new devices\IGBT-IRG4BC30K\Part 11\Turn On.csv
shape: (199, 2)
columns: ['2.000314E+0', '6.024289E-9']


...


 SMU Data for new devices\IGBT-IRG4BC30K\Part 14\LeakageIV.csv
shape: (199, 2)
columns: ['1.009896E+0', '4.306563E-9']


,1.009896E+0,4.306563E-9
0,4.013748,1.182948e-08
1,7.020504,1.792339e-08
2,10.066690,2.615520e-08



 SMU Data for new devices\IGBT-IRG4BC30K\Part 14\Turn On.csv
shape: (199, 2)
columns: ['2.000308E+0', '5.953778E-9']


,2.000308E+0,5.953778E-9
0,2.020158,5.907585e-09
1,2.040077,5.831455e-09
2,2.060678,5.759516e-09


,file,rows,columns,column_names,dtypes
0,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,199,2,5.452415E+2 | 1.042480E-5,5.452415E+2:float64 | 1.042480E-5:float64
1,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,199,2,1.007554E+0 | 3.848548E-9,1.007554E+0:float64 | 3.848548E-9:float64
2,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,199,2,2.000314E+0 | 6.024289E-9,2.000314E+0:float64 | 6.024289E-9:float64
3,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,199,2,5.451473E+2 | 1.042489E-5,5.451473E+2:float64 | 1.042489E-5:float64
4,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,199,2,1.007328E+0 | 5.611045E-9,1.007328E+0:float64 | 5.611045E-9:float64
5,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,199,2,2.000283E+0 | 5.188932E-9,2.000283E+0:float64 | 5.188932E-9:float64
6,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,199,2,5.453868E+2 | 1.042470E-5,5.453868E+2:float64 | 1.042470E-5:float64
7,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,199,2,1.007544E+0 | 3.870134E-9,1.007544E+0:float64 | 3.870134E-9:float64
8,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,199,2,2.000331E+0 | 5.741808E-9,2.000331E+0:float64 | 5.741808E-9:float64
9,SMU Data for new devices\IGBT-IRG4BC30K\Part 1...,199,2,5.457329E+2 | 1.042456E-5,5.457329E+2:float64 | 1.042456E-5:float64


In [3]:
# ============================================================
# 3. CHECK WHETHER ALL CSV FILES SHARE THE SAME SCHEMA
# ============================================================

schema_map = {}

for path in csv_files:
    try:
        df = pd.read_csv(path, nrows=5)
        cols = tuple(map(str, df.columns))
        schema_map.setdefault(cols, []).append(path)
    except Exception as e:
        pass

print("Unique CSV schemas:", len(schema_map))

for i, (cols, paths) in enumerate(schema_map.items(), start=1):
    print(f"\nSCHEMA {i} — {len(paths)} files")
    print("Columns:", cols)
    print("Example:", paths[0].relative_to(DATA_ROOT))

Unique CSV schemas: 120

SCHEMA 1 — 1 files
Columns: ('5.452415E+2', '1.042480E-5')
Example: SMU Data for new devices\IGBT-IRG4BC30K\Part 11\Breakdown.csv

SCHEMA 2 — 1 files
Columns: ('1.007554E+0', '3.848548E-9')
Example: SMU Data for new devices\IGBT-IRG4BC30K\Part 11\LeakageIV.csv

SCHEMA 3 — 1 files
Columns: ('2.000314E+0', '6.024289E-9')
Example: SMU Data for new devices\IGBT-IRG4BC30K\Part 11\Turn On.csv

SCHEMA 4 — 1 files
Columns: ('5.451473E+2', '1.042489E-5')
Example: SMU Data for new devices\IGBT-IRG4BC30K\Part 12\Breakdown.csv

SCHEMA 5 — 1 files
Columns: ('1.007328E+0', '5.611045E-9')
Example: SMU Data for new devices\IGBT-IRG4BC30K\Part 12\LeakageIV.csv

SCHEMA 6 — 1 files
Columns: ('2.000283E+0', '5.188932E-9')
Example: SMU Data for new devices\IGBT-IRG4BC30K\Part 12\Turn On.csv

SCHEMA 7 — 1 files
Columns: ('5.453868E+2', '1.042470E-5')
Example: SMU Data for new devices\IGBT-IRG4BC30K\Part 13\Breakdown.csv

SCHEMA 8 — 1 files
Columns: ('1.007544E+0', '3.870134E-9')
Exa

In [4]:
# ============================================================
# 4. NUMERIC SUMMARY OF REPRESENTATIVE CSV DATA
# ============================================================

if csv_files:
    sample_csv = csv_files[0]
    df_sample = pd.read_csv(sample_csv)

    print("Sample file:", sample_csv.relative_to(DATA_ROOT))
    print("Shape:", df_sample.shape)

    display(df_sample.describe(include="all").T)
    print("\nMissing values:")
    display(df_sample.isna().sum().to_frame("missing").T)

Sample file: SMU Data for new devices\IGBT-IRG4BC30K\Part 11\Breakdown.csv
Shape: (199, 2)


,count,mean,std,min,25%,50%,75%,max
5.452415E+2,199.0,600.290624,2.894268e+01,550.525700,575.433100,600.289900,625.168150,650.039400
1.042480E-5,199.0,0.000005,4.156637e-07,0.000004,0.000004,0.000005,0.000005,0.000005



Missing values:


,5.452415E+2,1.042480E-5
missing,0,0


## 5. MATLAB inspection helper

Some aging files contain ordinary numeric arrays, while the device-specific files contain a MATLAB `measurement` object. We therefore need a recursive inspector instead of assuming every `.mat` has flat arrays.

In [5]:
# ============================================================
# 5A. LOAD MATLAB FILES
# ============================================================

from scipy.io import loadmat

mat_files = sorted(DATA_ROOT.rglob("*.mat"))

print("MAT files:", len(mat_files))

for p in mat_files:
    print(" -", p.relative_to(DATA_ROOT))

MAT files: 12
 - Thermal Overstress Aging with DC at gate\20080429T135531.mat
 - Thermal Overstress Aging with Square Signal at gate\april22nd-23rdIgbtIRCG40BC30kd-A17.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 2\Device2  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 2\Device2b  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 2\Device2check  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 3\Device3  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 3\Device3b  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 3\Device3check  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 4\Device4  1.mat
 - Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 4\Device4

In [6]:
# ============================================================
# 5B. RECURSIVE OBJECT INSPECTOR
# ============================================================

def inspect_value(obj, name="root", depth=0, max_depth=4, rows=None):
    if rows is None:
        rows = []

    indent = "  " * depth
    obj_type = type(obj).__name__

    info = {
        "path": name,
        "type": obj_type,
        "shape": str(getattr(obj, "shape", None)),
        "dtype": str(getattr(obj, "dtype", None)),
        "size": getattr(obj, "size", None),
    }

    rows.append(info)

    if depth >= max_depth:
        return rows

    # MATLAB structs loaded with struct_as_record=False
    if hasattr(obj, "_fieldnames"):
        for field in obj._fieldnames:
            try:
                child = getattr(obj, field)
                inspect_value(child, f"{name}.{field}", depth + 1, max_depth, rows)
            except Exception as e:
                rows.append({
                    "path": f"{name}.{field}",
                    "type": f"ERROR: {e}",
                    "shape": "",
                    "dtype": "",
                    "size": ""
                })

    elif isinstance(obj, np.ndarray) and obj.dtype == object:
        # Inspect a small number of object elements.
        for idx, child in np.ndenumerate(obj):
            if idx[0] >= 3:
                break
            inspect_value(child, f"{name}[{idx}]", depth + 1, max_depth, rows)

    return rows


mat_structure_rows = []

for path in mat_files:
    print("\n" + "=" * 100)
    print(path.relative_to(DATA_ROOT))
    print("=" * 100)

    try:
        data = loadmat(path, squeeze_me=False, struct_as_record=False)

        for key, value in data.items():
            if not key.startswith("__"):
                rows = inspect_value(value, key, 0, 4)
                for r in rows:
                    r["file"] = str(path.relative_to(DATA_ROOT))
                mat_structure_rows.extend(rows)

        display(pd.DataFrame(mat_structure_rows[-100:]))

    except Exception as e:
        print("ERROR:", repr(e))

mat_structure_df = pd.DataFrame(mat_structure_rows)
mat_structure_df.to_csv("mat_nested_structure.csv", index=False)

print("\nSaved: mat_nested_structure.csv")


Thermal Overstress Aging with DC at gate\20080429T135531.mat


,path,type,shape,dtype,size,file
0,index,ndarray,"(1, 1)",int32,1,Thermal Overstress Aging with DC at gate\20080...
1,x,ndarray,"(1,)",<U18,1,Thermal Overstress Aging with DC at gate\20080...
2,gate_current,ndarray,"(1, 1)",float64,1,Thermal Overstress Aging with DC at gate\20080...
3,time,ndarray,"(1, 1)",float64,1,Thermal Overstress Aging with DC at gate\20080...
4,collector_current,ndarray,"(1, 1)",float64,1,Thermal Overstress Aging with DC at gate\20080...
5,heat_sink,ndarray,"(1, 1)",float64,1,Thermal Overstress Aging with DC at gate\20080...
6,package_temperture,ndarray,"(1, 1)",float64,1,Thermal Overstress Aging with DC at gate\20080...
7,gate_voltage,ndarray,"(1, 1)",float64,1,Thermal Overstress Aging with DC at gate\20080...
8,collector_voltage,ndarray,"(1, 1)",float64,1,Thermal Overstress Aging with DC at gate\20080...
9,GATE_VOLTAGE,ndarray,"(1, 301680)",float64,301680,Thermal Overstress Aging with DC at gate\20080...



Thermal Overstress Aging with Square Signal at gate\april22nd-23rdIgbtIRCG40BC30kd-A17.mat


,path,type,shape,dtype,size,file
0,"measurement[(0, 0)].steadyState[(0, 41727)].date",ndarray,"(1,)",<U21,1.0,Thermal Overstress Aging with Square Signal at...
1,"measurement[(0, 0)].steadyState[(0, 41727)].ti...",ndarray,"(1, 1)",float64,1.0,Thermal Overstress Aging with Square Signal at...
2,"measurement[(0, 0)].steadyState[(0, 41727)].ti...",ndarray,"(1, 1)",object,1.0,Thermal Overstress Aging with Square Signal at...
3,"measurement[(0, 0)].steadyState[(0, 41728)]",mat_struct,None,None,NaN,Thermal Overstress Aging with Square Signal at...
4,"measurement[(0, 0)].steadyState[(0, 41728)].date",ndarray,"(1,)",<U21,1.0,Thermal Overstress Aging with Square Signal at...
...,...,...,...,...,...,...
95,"measurement[(0, 0)].steadyState[(0, 41751)]",mat_struct,None,None,NaN,Thermal Overstress Aging with Square Signal at...
96,"measurement[(0, 0)].steadyState[(0, 41751)].date",ndarray,"(1,)",<U21,1.0,Thermal Overstress Aging with Square Signal at...
97,"measurement[(0, 0)].steadyState[(0, 41751)].ti...",ndarray,"(1, 1)",float64,1.0,Thermal Overstress Aging with Square Signal at...
98,"measurement[(0, 0)].steadyState[(0, 41751)].ti...",ndarray,"(1, 1)",object,1.0,Thermal Overstress Aging with Square Signal at...



Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 2\Device2  1.mat


...


Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 5\Device5  1.mat


,path,type,shape,dtype,size,file
0,"measurement[(0, 0)].steadyState[(0, 10486)]",mat_struct,None,None,NaN,Thermal Overstress Aging with Square Signal at...
1,"measurement[(0, 0)].steadyState[(0, 10486)].date",ndarray,"(1,)",<U22,1.0,Thermal Overstress Aging with Square Signal at...
2,"measurement[(0, 0)].steadyState[(0, 10486)].ti...",ndarray,"(1, 1)",float64,1.0,Thermal Overstress Aging with Square Signal at...
3,"measurement[(0, 0)].steadyState[(0, 10486)].ti...",ndarray,"(1, 1)",object,1.0,Thermal Overstress Aging with Square Signal at...
4,"measurement[(0, 0)].steadyState[(0, 10487)]",mat_struct,None,None,NaN,Thermal Overstress Aging with Square Signal at...
...,...,...,...,...,...,...
95,"measurement[(0, 0)].steadyState[(0, 10509)].ti...",ndarray,"(1, 1)",object,1.0,Thermal Overstress Aging with Square Signal at...
96,"measurement[(0, 0)].steadyState[(0, 10510)]",mat_struct,None,None,NaN,Thermal Overstress Aging with Square Signal at...
97,"measurement[(0, 0)].steadyState[(0, 10510)].date",ndarray,"(1,)",<U22,1.0,Thermal Overstress Aging with Square Signal at...
98,"measurement[(0, 0)].steadyState[(0, 10510)].ti...",ndarray,"(1, 1)",float64,1.0,Thermal Overstress Aging with Square Signal at...



Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 5\Device5check  1.mat


,path,type,shape,dtype,size,file
0,"measurement[(0, 0)].steadyState[(0, 137)]",mat_struct,None,None,NaN,Thermal Overstress Aging with Square Signal at...
1,"measurement[(0, 0)].steadyState[(0, 137)].date",ndarray,"(1,)",<U21,1.0,Thermal Overstress Aging with Square Signal at...
2,"measurement[(0, 0)].steadyState[(0, 137)].time...",ndarray,"(1, 1)",float64,1.0,Thermal Overstress Aging with Square Signal at...
3,"measurement[(0, 0)].steadyState[(0, 137)].time...",ndarray,"(1, 1)",object,1.0,Thermal Overstress Aging with Square Signal at...
4,"measurement[(0, 0)].steadyState[(0, 138)]",mat_struct,None,None,NaN,Thermal Overstress Aging with Square Signal at...
...,...,...,...,...,...,...
95,"measurement[(0, 0)].steadyState[(0, 160)].time...",ndarray,"(1, 1)",object,1.0,Thermal Overstress Aging with Square Signal at...
96,"measurement[(0, 0)].steadyState[(0, 161)]",mat_struct,None,None,NaN,Thermal Overstress Aging with Square Signal at...
97,"measurement[(0, 0)].steadyState[(0, 161)].date",ndarray,"(1,)",<U21,1.0,Thermal Overstress Aging with Square Signal at...
98,"measurement[(0, 0)].steadyState[(0, 161)].time...",ndarray,"(1, 1)",float64,1.0,Thermal Overstress Aging with Square Signal at...



Saved: mat_nested_structure.csv


In [7]:
# ============================================================
# 6. FOCUS ON DEVICE-SPECIFIC MEASUREMENT OBJECTS
# ============================================================

device_mat_files = [
    p for p in mat_files
    if "Aging Data" in str(p)
]

print("Device aging MAT files:", len(device_mat_files))

for path in device_mat_files:
    print("\nFILE:", path.relative_to(DATA_ROOT))

    try:
        data = loadmat(path, squeeze_me=True, struct_as_record=False)

        if "measurement" in data:
            measurement = data["measurement"]

            print("measurement type:", type(measurement))
            print("measurement shape:", getattr(measurement, "shape", None))
            print("measurement dtype:", getattr(measurement, "dtype", None))

            if hasattr(measurement, "_fieldnames"):
                print("measurement fields:")
                for field in measurement._fieldnames:
                    value = getattr(measurement, field)
                    print(
                        f"  {field:35s}",
                        "type=", type(value).__name__,
                        "shape=", getattr(value, "shape", None),
                        "dtype=", getattr(value, "dtype", None)
                    )
            else:
                print("No _fieldnames attribute. Inspecting object recursively:")
                display(pd.DataFrame(
                    inspect_value(measurement, "measurement", 0, 5)
                ))

    except Exception as e:
        print("ERROR:", repr(e))

Device aging MAT files: 10

FILE: Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 2\Device2  1.mat
measurement type: <class 'scipy.io.matlab._mio5_params.mat_struct'>
measurement shape: None
measurement dtype: None
measurement fields:
  pwmTempControllerState              type= ndarray shape= (151,) dtype= object
  transient                           type= ndarray shape= (110,) dtype= object
  steadyState                         type= ndarray shape= (10978,) dtype= object

FILE: Thermal Overstress Aging with Square Signal at gate and SMU data\Aging Data\Device 2\Device2b  1.mat
measurement type: <class 'scipy.io.matlab._mio5_params.mat_struct'>
measurement shape: None
measurement dtype: None
measurement fields:
  pwmTempControllerState              type= ndarray shape= (103,) dtype= object
  transient                           type= ndarray shape= (65,) dtype= object
  steadyState                         type= ndarray shape= (6435,) dtype= object

FIL

## 7. Inspect the aging log

The Excel log is potentially very important because it may tell us which device was aged when, under what condition, and when measurements were taken.

We will use it as **metadata/ground truth for analysis**, not automatically as a supervised target.

In [8]:
# ============================================================
# 7. AGING LOG
# ============================================================

xls_files = list(DATA_ROOT.rglob("*.xls")) + list(DATA_ROOT.rglob("*.xlsx"))

for path in xls_files:
    print("\nFILE:", path.relative_to(DATA_ROOT))

    try:
        xl = pd.ExcelFile(path)
        print("Sheets:", xl.sheet_names)

        for sheet in xl.sheet_names:
            df_log = pd.read_excel(path, sheet_name=sheet)

            print("\nSHEET:", sheet)
            print("Shape:", df_log.shape)
            print("Columns:", list(df_log.columns))

            display(df_log)
    except Exception as e:
        print("ERROR:", repr(e))


FILE: Thermal Overstress Aging with Square Signal at gate and SMU data\Device Aging log.xls
ERROR: ImportError("Missing optional dependency 'xlrd'. Install xlrd >= 2.0.1 for xls Excel support Use pip or conda to install xlrd.")


In [9]:
# ============================================================
# 8. IDENTIFY CANDIDATE SIGNAL NAMES
# ============================================================

signal_keywords = [
    "time", "timestamp", "cycle",
    "gate", "vge", "voltage",
    "collector", "current", "ic",
    "temperature", "temp", "heat",
    "power", "resistance",
    "device", "aging", "age", "failure"
]

candidate_fields = []

# CSV fields
for cols in schema_map.keys():
    for c in cols:
        candidate_fields.append(("CSV", c))

# MATLAB fields from the flat DC experiment
if not mat_structure_df.empty:
    for p in mat_structure_df["path"].dropna().unique():
        candidate_fields.append(("MAT", p))

candidate_df = pd.DataFrame(candidate_fields, columns=["source", "field"])
candidate_df["keywords"] = candidate_df["field"].astype(str).str.lower().apply(
    lambda x: ", ".join(k for k in signal_keywords if k in x)
)

display(candidate_df[candidate_df["keywords"] != ""].drop_duplicates())
candidate_df.to_csv("candidate_signal_map.csv", index=False)

,source,field,keywords
242,MAT,gate_current,"gate, current"
243,MAT,time,time
244,MAT,collector_current,"collector, current"
245,MAT,heat_sink,heat
246,MAT,package_temperture,"temp, age"
...,...,...,...
195664,MAT,"measurement[(0, 0)].transient[(0, 142)].timeEpoch",time
195665,MAT,"measurement[(0, 0)].transient[(0, 143)].timeEpoch",time
195666,MAT,"measurement[(0, 0)].transient[(0, 144)].timeEpoch",time
195667,MAT,"measurement[(0, 0)].transient[(0, 145)].timeEpoch",time


# Decision point

After running this notebook, STOP.

We will use the outputs to decide:

1. which experiment is the main BurnInWatch source
2. which signals are comparable across devices
3. whether the aging data can be represented as time windows
4. how the 120 new-device SMU CSVs should be used
5. what the canonical CSV schema should be
6. how to construct an unsupervised normal-vs-anomalous evaluation without leaking device identity or future measurements

**Do not train Isolation Forest yet.**